# BS Commitment Attribution Patching with NNsight

Barebones version of the old IOI notebook, but using matched BS commitment branches:

- deceptive branch: `x_D = p + s_D`
- truthful branch: `x_H = p + s_H`

where both branches share the same prefix `p`, commitment examples are filtered by
`Delta_k > 0.3`, and the default objective restores truthfulness with
`score(s_H | p) - score(s_D | p)`. By default `score` is
`exp(mean token logprob)`, a stable probability-like sentence score.
Set `ATTR_PATCH_SENTENCE_SCORE=mean_logprob` to recover the old objective, or
`ATTR_PATCH_OBJECTIVE=deception` to use the opposite orientation.

This notebook keeps the `nnsight` flow simple and only does attribution patching over
attention-head, MLP-output, and residual-stream sites.


In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gc
import importlib
import inspect
import json
import math
import re
import sys
from pathlib import Path

import einops
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import torch
import torch.nn.functional as F
from IPython.display import Markdown, clear_output, display
import nnsight
from nnsight import LanguageModel

REPO_ROOT = Path("/playpen-ssd/smerrill/deception2")
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import activation_patching as ap
from activation_patching import encode_text_for_model, resolve_decoder_layers
import activation_patching_debug as apd
apd = importlib.reload(apd)

pio.renderers.default = "plotly_mimetype+notebook_connected+notebook"
pd.options.display.max_colwidth = 160
print("nnsight", nnsight.__version__)


In [ ]:
def latest_snapshot_path(root: Path) -> Path | None:
    snapshot_root = root / "snapshots"
    if not snapshot_root.exists():
        return None
    snapshots = sorted(path for path in snapshot_root.iterdir() if path.is_dir())
    return snapshots[-1] if snapshots else None


MODEL_ID = os.environ.get("ATTR_PATCH_MODEL_ID", "gpt-oss-20b").strip()
MODEL_CONFIGS = {
    "gpt-oss-20b": {
        "hf_repo": "openai/gpt-oss-20b",
        "hf_cache_dir": "models--openai--gpt-oss-20b",
    },
    "DeepSeek-R1-Distill-Qwen-7B": {
        "hf_repo": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
        "hf_cache_dir": "models--deepseek-ai--DeepSeek-R1-Distill-Qwen-7B",
    },
}
if MODEL_ID not in MODEL_CONFIGS:
    raise ValueError(f"Unsupported ATTR_PATCH_MODEL_ID={MODEL_ID!r}. Choose from {sorted(MODEL_CONFIGS)}.")

model_cfg = MODEL_CONFIGS[MODEL_ID]
hf_cache_root = Path("/playpen-ssd/smerrill/huggingface/transformers")
cached_snapshot = latest_snapshot_path(hf_cache_root / model_cfg["hf_cache_dir"])
LOCALIZATION_DIR = Path(
    os.environ.get(
        "ATTR_PATCH_LOCALIZATION_DIR",
        str(REPO_ROOT / "DatasetMain" / "bs" / MODEL_ID / "localization"),
    )
)
LOCAL_MODEL_SNAPSHOT = Path(
    os.environ.get(
        "ATTR_PATCH_MODEL_SNAPSHOT",
        str(cached_snapshot or model_cfg["hf_repo"]),
    )
)
MODEL_NAME = str(
    LOCAL_MODEL_SNAPSHOT if LOCAL_MODEL_SNAPSHOT.exists() else model_cfg["hf_repo"]
)
PAIR_CACHE_PATH = Path(
    os.environ.get(
        "ATTR_PATCH_PAIR_CACHE_PATH",
        str(REPO_ROOT / "Cache" / "activation_patching" / f"bs_commitment_pairs_for_notebook__{MODEL_ID}.jsonl"),
    )
)
DTYPE_BY_NAME = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}
DTYPE_NAME = os.environ.get("ATTR_PATCH_DTYPE", "bfloat16").strip().lower()
if DTYPE_NAME not in DTYPE_BY_NAME:
    raise ValueError(f"Unsupported ATTR_PATCH_DTYPE={DTYPE_NAME!r}. Choose from {sorted(DTYPE_BY_NAME)}.")
MODEL_DTYPE = DTYPE_BY_NAME[DTYPE_NAME]
PAIR_COUNT = int(os.environ.get("ATTR_PATCH_PAIR_COUNT", "10"))
BATCH_PAIR_COUNT = int(os.environ.get("ATTR_PATCH_BATCH_PAIR_COUNT", "1"))
PAIR_SEARCH_LIMIT = int(os.environ.get("ATTR_PATCH_PAIR_SEARCH_LIMIT", "128"))
OBJECTIVE_TARGET = os.environ.get("ATTR_PATCH_OBJECTIVE", "truthfulness").strip().lower()
if OBJECTIVE_TARGET not in {"truthfulness", "deception"}:
    raise ValueError("ATTR_PATCH_OBJECTIVE must be 'truthfulness' or 'deception'.")
SENTENCE_SCORE_MODE = os.environ.get("ATTR_PATCH_SENTENCE_SCORE", "geomean_prob").strip().lower()
SENTENCE_SCORE_MODES = {"mean_logprob", "sum_logprob", "geomean_prob", "sentence_prob"}
if SENTENCE_SCORE_MODE not in SENTENCE_SCORE_MODES:
    raise ValueError(f"ATTR_PATCH_SENTENCE_SCORE must be one of {sorted(SENTENCE_SCORE_MODES)}.")
SITE_FAMILIES_RAW = os.environ.get("ATTR_PATCH_SITE_FAMILIES", "all").strip().lower()
if SITE_FAMILIES_RAW in {"", "all"}:
    SITE_FAMILIES = ("attn_heads", "mlp", "resid")
else:
    site_family_aliases = {
        "attention": "attn_heads",
        "attention_heads": "attn_heads",
        "attn": "attn_heads",
        "attn_head": "attn_heads",
        "attn_heads": "attn_heads",
        "mlp": "mlp",
        "mlps": "mlp",
        "resid": "resid",
        "residual": "resid",
        "residual_stream": "resid",
    }
    SITE_FAMILIES = tuple(
        dict.fromkeys(
            site_family_aliases.get(part.strip(), part.strip())
            for part in SITE_FAMILIES_RAW.split(",")
            if part.strip()
        )
    )
unknown_site_families = sorted(set(SITE_FAMILIES) - {"attn_heads", "mlp", "resid"})
if unknown_site_families:
    raise ValueError(f"Unsupported ATTR_PATCH_SITE_FAMILIES entries: {unknown_site_families}")
MIN_COMMITMENT_DELTA = float(os.environ.get("ATTR_PATCH_MIN_COMMITMENT_DELTA", "0.3"))
MIN_COMMITMENT_DECEPTION_RATE = float(os.environ.get("ATTR_PATCH_MIN_COMMITMENT_DECEPTION_RATE", "0.0"))
MIN_DONOR_CLARITY_SCORE = float(os.environ.get("ATTR_PATCH_MIN_DONOR_CLARITY_SCORE", "0.0"))
MIN_NUM_VALID = int(os.environ.get("ATTR_PATCH_MIN_NUM_VALID", "11"))
MIN_SENTENCE_ALPHA_WORDS = int(os.environ.get("ATTR_PATCH_MIN_SENTENCE_ALPHA_WORDS", "4"))
EXCLUDE_MULTILINE_SENTENCES = os.environ.get("ATTR_PATCH_EXCLUDE_MULTILINE_SENTENCES", "1") == "1"
MAX_INPUT_TOKENS_RAW = os.environ.get("ATTR_PATCH_MAX_INPUT_TOKENS", "auto").strip()
if MAX_INPUT_TOKENS_RAW.lower() in {"", "auto", "dataset", "max", "none"}:
    MAX_INPUT_TOKENS = None
else:
    MAX_INPUT_TOKENS = int(MAX_INPUT_TOKENS_RAW)


In [ ]:
model = LanguageModel(
    MODEL_NAME,
    device_map="auto",
    dispatch=True,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=True,
)
clear_output()
print(model)

tokenizer = model.tokenizer
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

layers, layer_path = resolve_decoder_layers(model)
n_layers = len(layers)
n_heads = int(model.config.num_attention_heads)
head_dim = int(getattr(model.config, "head_dim", 0) or (model.config.hidden_size // n_heads))
ACTIVATION_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


def attn_out_input(layer):
    if hasattr(layer, "self_attn") and hasattr(layer.self_attn, "o_proj"):
        return layer.self_attn.o_proj.input
    if hasattr(layer, "attn") and hasattr(layer.attn, "c_proj"):
        return layer.attn.c_proj.input
    raise AttributeError("Could not find the attention output projection input for this layer.")


def first_tensor_output(module_or_layer, *, force_first=False):
    output = module_or_layer.output
    if force_first:
        return output[0]
    fake_value = getattr(getattr(output, "node", None), "fake_value", None)
    if isinstance(fake_value, tuple):
        return output[0]
    return output


def module_output_is_tuple(module_or_layer):
    class_name = module_or_layer._module.__class__.__name__ if hasattr(module_or_layer, "_module") else module_or_layer.__class__.__name__
    return class_name in {"GptOssMLP"} or "GptOss" in class_name


def mlp_output(layer):
    if hasattr(layer, "mlp"):
        return first_tensor_output(layer.mlp, force_first=module_output_is_tuple(layer.mlp))
    if hasattr(layer, "feed_forward"):
        return first_tensor_output(layer.feed_forward, force_first=module_output_is_tuple(layer.feed_forward))
    raise AttributeError("Could not find an MLP output for this layer.")


def resid_output(layer):
    return first_tensor_output(layer)


NO_HEAD = -1


def site_key(site_type, layer_idx, head_idx=NO_HEAD):
    return (str(site_type), int(layer_idx), int(head_idx))


def site_label(site):
    site_type, layer_idx, head_idx = site
    if site_type == "attn_head":
        if int(head_idx) < 0:
            return f"L{int(layer_idx):02d}_attn_heads"
        return f"L{int(layer_idx):02d}H{int(head_idx):02d}"
    return f"L{int(layer_idx):02d}_{site_type}"


print(layer_path, "|", n_layers, "layers |", n_heads, "heads")
display(
    Markdown(
        "\n".join(
            [
                f"- `MODEL_ID`: `{MODEL_ID}`",
                f"- `MODEL_NAME`: `{MODEL_NAME}`",
                f"- `MODEL_DTYPE`: `{DTYPE_NAME}`",
                f"- `OBJECTIVE_TARGET`: `{OBJECTIVE_TARGET}`",
                f"- `SENTENCE_SCORE_MODE`: `{SENTENCE_SCORE_MODE}`",
                f"- `SITE_FAMILIES`: `{','.join(SITE_FAMILIES)}`",
                f"- `PAIR_COUNT`: `{PAIR_COUNT}`",
                f"- `BATCH_PAIR_COUNT`: `{BATCH_PAIR_COUNT}`",
                f"- `MIN_NUM_VALID`: `{MIN_NUM_VALID}`",
                f"- `MIN_SENTENCE_ALPHA_WORDS`: `{MIN_SENTENCE_ALPHA_WORDS}`",
                f"- `EXCLUDE_MULTILINE_SENTENCES`: `{EXCLUDE_MULTILINE_SENTENCES}`",
                f"- `MAX_INPUT_TOKENS_RAW`: `{MAX_INPUT_TOKENS_RAW}`",
                f"- `LOCALIZATION_DIR`: `{LOCALIZATION_DIR}`",
                f"- `PAIR_CACHE_PATH`: `{PAIR_CACHE_PATH}`",
                f"- `PYTORCH_CUDA_ALLOC_CONF`: `{os.environ.get('PYTORCH_CUDA_ALLOC_CONF', '(unset)')}`",
            ]
        )
    )
)


In [ ]:
load_commitment_pairs = apd.load_commitment_pairs
load_commitment_pairs_kwargs = dict(
    localization_dir=LOCALIZATION_DIR,
    pair_cache_path=PAIR_CACHE_PATH,
    pair_count=PAIR_COUNT,
    pair_search_limit=PAIR_SEARCH_LIMIT,
    refresh_cache=False,
    min_commitment_delta=MIN_COMMITMENT_DELTA,
    min_commitment_deception_rate=MIN_COMMITMENT_DECEPTION_RATE,
    min_donor_clarity_score=MIN_DONOR_CLARITY_SCORE,
    disable_tqdm=False,
)
load_commitment_pairs_signature = inspect.signature(load_commitment_pairs)
if "min_num_valid" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["min_num_valid"] = MIN_NUM_VALID
if "min_sentence_alpha_words" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["min_sentence_alpha_words"] = MIN_SENTENCE_ALPHA_WORDS
if "exclude_multiline_sentences" in load_commitment_pairs_signature.parameters:
    load_commitment_pairs_kwargs["exclude_multiline_sentences"] = EXCLUDE_MULTILINE_SENTENCES

pairs_df = load_commitment_pairs(**load_commitment_pairs_kwargs).copy()

display(
    pairs_df[
        [
            "pair_index",
            "example_id",
            "commitment_delta",
            "shared_context_num_valid",
            "deceptive_prefix_num_valid",
            "donor_clarity_score",
            "n_truthful_donors",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
        ]
    ]
)


In [ ]:
def encode_branch(prefix_text, full_text):
    prefix_ids = encode_text_for_model(
        tokenizer,
        prefix_text,
        max_input_tokens=MAX_INPUT_TOKENS,
    )["input_ids"][0]
    full_ids = encode_text_for_model(
        tokenizer,
        full_text,
        max_input_tokens=MAX_INPUT_TOKENS,
    )["input_ids"][0]
    prefix_len = int(prefix_ids.shape[0])
    total_len = int(full_ids.shape[0])
    if total_len <= prefix_len:
        raise ValueError("Each branch needs at least one token after the shared prefix.")
    if not torch.equal(full_ids[:prefix_len], prefix_ids):
        raise ValueError("Full branch tokenization does not start with the shared prefix tokens.")
    return {
        "full_ids": full_ids,
        "prefix_len": prefix_len,
        "total_len": total_len,
        "score_start_pos": prefix_len,
        "score_stop_pos": total_len,
    }


def prepare_pair_records(pairs):
    prepared = []
    for pair_row in pairs.to_dict(orient="records"):
        deceptive_encoded = encode_branch(
            pair_row["shared_prefix_text"],
            pair_row["deceptive_branch_text"],
        )
        truthful_encoded = encode_branch(
            pair_row["shared_prefix_text"],
            pair_row["truthful_branch_text"],
        )
        if int(deceptive_encoded["prefix_len"]) != int(truthful_encoded["prefix_len"]):
            raise ValueError("Prefix token lengths differ across deceptive/truthful branches.")
        prepared.append(
            {
                **pair_row,
                "deceptive_encoded": deceptive_encoded,
                "truthful_encoded": truthful_encoded,
                "prefix_token_len": int(deceptive_encoded["prefix_len"]),
                "deceptive_total_len": int(deceptive_encoded["total_len"]),
                "truthful_total_len": int(truthful_encoded["total_len"]),
                "max_total_len": int(
                    max(
                        deceptive_encoded["total_len"],
                        truthful_encoded["total_len"],
                    )
                ),
            }
        )
    return sorted(
        prepared,
        key=lambda row: (int(row["max_total_len"]), int(row["pair_index"])),
    )


def pair_chunk_slices(prepared_pairs, batch_pair_count):
    if int(batch_pair_count) <= 0:
        raise ValueError("BATCH_PAIR_COUNT must be positive.")
    return [
        prepared_pairs[start : start + int(batch_pair_count)]
        for start in range(0, len(prepared_pairs), int(batch_pair_count))
    ]


def make_row(prepared_pair, branch_role):
    if branch_role == "deceptive":
        encoded = prepared_pair["deceptive_encoded"]
        sentence_text = prepared_pair["deceptive_commitment_sentence"]
    elif branch_role == "truthful":
        encoded = prepared_pair["truthful_encoded"]
        sentence_text = prepared_pair["truthful_donor_sentence"]
    else:
        raise ValueError(f"Unsupported branch_role={branch_role!r}")
    return {
        "pair_index": int(prepared_pair["pair_index"]),
        "example_id": str(prepared_pair["example_id"]),
        "branch_role": branch_role,
        "sentence_text": sentence_text,
        **encoded,
    }


def build_rows(prepared_pairs):
    clean_rows = []
    corrupted_rows = []
    for prepared_pair in prepared_pairs:
        deceptive_row = make_row(prepared_pair, "deceptive")
        truthful_row = make_row(prepared_pair, "truthful")
        if OBJECTIVE_TARGET == "truthfulness":
            clean_rows.extend([truthful_row, deceptive_row])
            corrupted_rows.extend([deceptive_row.copy(), truthful_row.copy()])
        else:
            clean_rows.extend([deceptive_row, truthful_row])
            corrupted_rows.extend([truthful_row.copy(), deceptive_row.copy()])
    return clean_rows, corrupted_rows


def pad_rows(rows):
    max_len = max(int(row["total_len"]) for row in rows)
    input_ids = torch.full((len(rows), max_len), tokenizer.pad_token_id, dtype=torch.long)
    attention_mask = torch.zeros((len(rows), max_len), dtype=torch.long)
    for row_idx, row in enumerate(rows):
        total_len = int(row["total_len"])
        input_ids[row_idx, :total_len] = row["full_ids"]
        attention_mask[row_idx, :total_len] = 1
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "rows": rows,
    }


def build_batches_for_pairs(prepared_pairs):
    clean_rows, corrupted_rows = build_rows(prepared_pairs)
    clean_batch = pad_rows(clean_rows)
    corrupted_batch = pad_rows(corrupted_rows)
    clean_inputs = {
        "input_ids": clean_batch["input_ids"],
        "attention_mask": clean_batch["attention_mask"],
    }
    corrupted_inputs = {
        "input_ids": corrupted_batch["input_ids"],
        "attention_mask": corrupted_batch["attention_mask"],
    }
    return clean_batch, corrupted_batch, clean_inputs, corrupted_inputs


prepared_pairs = prepare_pair_records(pairs_df)
DATASET_MAX_INPUT_TOKENS = max(int(pair["max_total_len"]) for pair in prepared_pairs)
EFFECTIVE_MAX_INPUT_TOKENS = (
    int(DATASET_MAX_INPUT_TOKENS) if MAX_INPUT_TOKENS is None else int(MAX_INPUT_TOKENS)
)
MAX_INPUT_TOKENS = int(EFFECTIVE_MAX_INPUT_TOKENS)
pair_chunks = pair_chunk_slices(prepared_pairs, BATCH_PAIR_COUNT)
total_pairs = len(prepared_pairs)

pairs_overview_df = pd.DataFrame(
    [
        {
            "pair_index": int(pair["pair_index"]),
            "example_id": str(pair["example_id"]),
            "batch_index": int(batch_idx),
            "prefix_token_len": int(pair["prefix_token_len"]),
            "deceptive_total_len": int(pair["deceptive_total_len"]),
            "truthful_total_len": int(pair["truthful_total_len"]),
            "max_total_len": int(pair["max_total_len"]),
            "shared_context_num_valid": int(pair["shared_context_num_valid"]),
            "deceptive_prefix_num_valid": int(pair["deceptive_prefix_num_valid"]),
            "commitment_delta": float(pair["commitment_delta"]),
        }
        for batch_idx, chunk in enumerate(pair_chunks)
        for pair in chunk
    ]
).sort_values("pair_index").reset_index(drop=True)

display(
    pairs_overview_df[
        [
            "pair_index",
            "example_id",
            "batch_index",
            "prefix_token_len",
            "deceptive_total_len",
            "truthful_total_len",
            "max_total_len",
            "shared_context_num_valid",
            "deceptive_prefix_num_valid",
            "commitment_delta",
        ]
    ]
)

preview_clean_batch, _, _, _ = build_batches_for_pairs(pair_chunks[0])
display(
    pd.DataFrame(
        [
            {
                "row_idx": row_idx,
                "pair_index": row["pair_index"],
                "branch_role": row["branch_role"],
                "prefix_len": row["prefix_len"],
                "total_len": row["total_len"],
                "sentence_text": row["sentence_text"],
            }
            for row_idx, row in enumerate(preview_clean_batch["rows"])
        ]
    )
)
print(
    f"Prepared {total_pairs} pair(s) across {len(pair_chunks)} chunk(s) "
    f"with batch_pair_count={BATCH_PAIR_COUNT}."
)
print(
    f"Effective max input tokens: {MAX_INPUT_TOKENS} "
    f"(dataset max needed: {DATASET_MAX_INPUT_TOKENS})"
)


In [ ]:
def clear_memory():
    if hasattr(model, "clear_edits"):
        model.clear_edits()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def saved_value(x):
    return getattr(x, "value", x)


def to_activation_device(source, activation_slice):
    return source.to(device=activation_slice.device, dtype=activation_slice.dtype)


def scored_logits_and_targets(logits, batch):
    scored_logits = []
    scored_targets = []
    row_lengths = []
    for row_idx, row in enumerate(batch["rows"]):
        start = int(row["score_start_pos"])
        stop = int(row["score_stop_pos"])
        row_logits = logits[row_idx, start - 1 : stop - 1, :]
        row_targets = batch["input_ids"][row_idx, start:stop].to(logits.device)
        scored_logits.append(row_logits)
        scored_targets.append(row_targets)
        row_lengths.append(int(stop - start))
    return torch.cat(scored_logits, dim=0), torch.cat(scored_targets, dim=0), row_lengths


def score_sentence_token_log_probs(row_token_log_probs):
    row_token_log_probs = row_token_log_probs.float()
    if SENTENCE_SCORE_MODE == "mean_logprob":
        return row_token_log_probs.mean()
    if SENTENCE_SCORE_MODE == "sum_logprob":
        return row_token_log_probs.sum()
    if SENTENCE_SCORE_MODE == "geomean_prob":
        return torch.exp(row_token_log_probs.mean())
    if SENTENCE_SCORE_MODE == "sentence_prob":
        return torch.exp(row_token_log_probs.sum())
    raise ValueError(f"Unsupported SENTENCE_SCORE_MODE={SENTENCE_SCORE_MODE!r}.")


def sentence_score_by_row(logits, batch):
    flat_logits, flat_targets, row_lengths = scored_logits_and_targets(logits, batch)
    token_log_probs = -F.cross_entropy(
        flat_logits,
        flat_targets,
        reduction="none",
    )
    scores = []
    offset = 0
    for row_len in row_lengths:
        row_token_log_probs = token_log_probs[offset : offset + row_len]
        scores.append(score_sentence_token_log_probs(row_token_log_probs))
        offset += row_len
    return torch.stack(scores)


def paired_metric_from_row_scores(row_scores):
    return (row_scores[0::2] - row_scores[1::2]).mean()


def paired_metric_from_logits(logits, batch):
    row_scores = sentence_score_by_row(logits, batch)
    return paired_metric_from_row_scores(row_scores)


def objective_margin_from_scores(score_d, score_h):
    if OBJECTIVE_TARGET == "truthfulness":
        return float(score_h) - float(score_d)
    return float(score_d) - float(score_h)


summary_records = []
clean_margin_total = 0.0
corrupted_margin_total = 0.0

for chunk_idx, chunk_pairs in enumerate(pair_chunks, start=1):
    clean_batch, corrupted_batch, clean_inputs, corrupted_inputs = build_batches_for_pairs(chunk_pairs)

    with torch.inference_mode():
        clean_logits = model.trace(
            clean_inputs,
            trace=False,
        ).logits
    clean_row_scores = sentence_score_by_row(clean_logits, clean_batch)
    clean_margin = float(paired_metric_from_row_scores(clean_row_scores).item())
    clean_row_scores = clean_row_scores.detach().cpu()
    del clean_logits
    clear_memory()

    with torch.inference_mode():
        corrupted_logits = model.trace(
            corrupted_inputs,
            trace=False,
        ).logits
    corrupted_row_scores = sentence_score_by_row(corrupted_logits, corrupted_batch)
    corrupted_margin = float(paired_metric_from_row_scores(corrupted_row_scores).item())
    del corrupted_row_scores
    del corrupted_logits
    clear_memory()

    chunk_pair_count = len(chunk_pairs)
    clean_margin_total += clean_margin * float(chunk_pair_count)
    corrupted_margin_total += corrupted_margin * float(chunk_pair_count)

    for local_idx, pair_row in enumerate(chunk_pairs):
        row_start = 2 * local_idx
        score_by_role = {
            str(clean_batch["rows"][row_start + offset]["branch_role"]): float(
                clean_row_scores[row_start + offset].item()
            )
            for offset in range(2)
        }
        score_d = float(score_by_role["deceptive"])
        score_h = float(score_by_role["truthful"])
        objective_metric = objective_margin_from_scores(score_d, score_h)
        summary_records.append(
            {
                "pair_index": int(pair_row["pair_index"]),
                "example_id": str(pair_row["example_id"]),
                "commitment_delta": float(pair_row["commitment_delta"]),
                "max_total_len": int(pair_row["max_total_len"]),
                "score_D": score_d,
                "score_H": score_h,
                "deceptive_minus_truthful": score_d - score_h,
                "truthful_minus_deceptive": score_h - score_d,
                "metric": objective_metric,
                "deceptive_commitment_sentence": str(pair_row["deceptive_commitment_sentence"]),
                "truthful_donor_sentence": str(pair_row["truthful_donor_sentence"]),
            }
        )

    print(
        f"Scored chunk {chunk_idx}/{len(pair_chunks)} | pairs={chunk_pair_count} | "
        f"max_total_len={max(int(pair['max_total_len']) for pair in chunk_pairs)}"
    )

    del clean_row_scores, clean_batch, corrupted_batch, clean_inputs, corrupted_inputs
    clear_memory()

CLEAN_BASELINE = clean_margin_total / float(total_pairs)
CORRUPTED_BASELINE = corrupted_margin_total / float(total_pairs)

if abs(CLEAN_BASELINE - CORRUPTED_BASELINE) < 1e-8:
    raise ValueError("Clean and corrupted baselines are too similar for normalization.")


def teacher_forced_metric(logits, batch):
    raw_metric = paired_metric_from_logits(logits, batch)
    return (raw_metric - CORRUPTED_BASELINE) / (CLEAN_BASELINE - CORRUPTED_BASELINE)


summary_df = pd.DataFrame(summary_records).sort_values("pair_index").reset_index(drop=True)
summary_df["clean_favors_deceptive"] = summary_df["deceptive_minus_truthful"] > 0
summary_df["clean_favors_truthful"] = summary_df["truthful_minus_deceptive"] > 0
summary_df["clean_favors_objective"] = summary_df["metric"] > 0
summary_df["abs_metric"] = summary_df["metric"].abs()

metric_sanity_df = pd.DataFrame(
    [
        {
            "n_pairs": int(len(summary_df)),
            "objective_target": OBJECTIVE_TARGET,
            "sentence_score_mode": SENTENCE_SCORE_MODE,
            "mean_score_D": float(summary_df["score_D"].mean()),
            "mean_score_H": float(summary_df["score_H"].mean()),
            "mean_deceptive_minus_truthful": float(summary_df["deceptive_minus_truthful"].mean()),
            "mean_truthful_minus_deceptive": float(summary_df["truthful_minus_deceptive"].mean()),
            "mean_objective_margin": float(summary_df["metric"].mean()),
            "median_objective_margin": float(summary_df["metric"].median()),
            "frac_clean_favors_deceptive": float(summary_df["clean_favors_deceptive"].mean()),
            "frac_clean_favors_truthful": float(summary_df["clean_favors_truthful"].mean()),
            "frac_clean_favors_objective": float(summary_df["clean_favors_objective"].mean()),
            "clean_baseline_b": float(CLEAN_BASELINE),
            "corrupted_baseline_b_prime": float(CORRUPTED_BASELINE),
            "denominator_b_minus_b_prime": float(CLEAN_BASELINE - CORRUPTED_BASELINE),
        }
    ]
)

display(
    summary_df[
        [
            "pair_index",
            "example_id",
            "commitment_delta",
            "score_D",
            "score_H",
            "metric",
            "deceptive_minus_truthful",
            "truthful_minus_deceptive",
            "clean_favors_deceptive",
            "clean_favors_truthful",
            "clean_favors_objective",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
        ]
    ]
)
display(metric_sanity_df)

if CLEAN_BASELINE < CORRUPTED_BASELINE:
    display(
        Markdown(
            "**Sanity warning:** the clean objective baseline is below the corrupted baseline. "
            "This means the current clean/corrupted ordering is still not aligned with the selected "
            "`OBJECTIVE_TARGET`; attribution and faithfulness will be hard to interpret until this "
            "orientation is fixed."
        )
    )
    display(
        summary_df.sort_values("metric")[
            [
                "pair_index",
                "metric",
                "score_D",
                "score_H",
                "deceptive_minus_truthful",
                "truthful_minus_deceptive",
                "deceptive_commitment_sentence",
                "truthful_donor_sentence",
            ]
        ].head(10)
    )
else:
    display(
        Markdown(
            f"**Sanity check passed:** clean is above corrupted for the `{OBJECTIVE_TARGET}` "
            "teacher-forced objective."
        )
    )

print(f"Clean baseline: {CLEAN_BASELINE:.4f}")
print(f"Corrupted baseline: {CORRUPTED_BASELINE:.4f}")
print("Clean metric -> 1.0000")
print("Corrupted metric -> 0.0000")


In [ ]:
attribution_sums = {}
for layer_idx in range(n_layers):
    if "attn_heads" in SITE_FAMILIES:
        for head_idx in range(n_heads):
            attribution_sums[site_key("attn_head", layer_idx, head_idx)] = 0.0
    if "mlp" in SITE_FAMILIES:
        attribution_sums[site_key("mlp", layer_idx)] = 0.0
    if "resid" in SITE_FAMILIES:
        attribution_sums[site_key("resid", layer_idx)] = 0.0

diag_records = []

print("The traced objective below is expected to stay near 0.0 because each pass starts from the corrupted baseline.")


def attribution_proxy_for_family(layer, family):
    if family == "attn_heads":
        return attn_out_input(layer)
    if family == "mlp":
        return mlp_output(layer)
    if family == "resid":
        return resid_output(layer)
    raise ValueError(f"Unsupported site family: {family}")


for chunk_idx, chunk_pairs in enumerate(pair_chunks, start=1):
    clean_batch, corrupted_batch, clean_inputs, corrupted_inputs = build_batches_for_pairs(chunk_pairs)
    chunk_pair_count = len(chunk_pairs)
    chunk_max_total_len = max(int(pair["max_total_len"]) for pair in chunk_pairs)
    print(
        f"Chunk {chunk_idx}/{len(pair_chunks)} | pairs={chunk_pair_count} | "
        f"max_total_len={chunk_max_total_len}"
    )
    with torch.inference_mode():
        corrupted_logits_for_chunk = model.trace(corrupted_inputs, trace=False).logits
    chunk_unpatched_value = float(teacher_forced_metric(corrupted_logits_for_chunk, corrupted_batch).item())
    del corrupted_logits_for_chunk
    clear_memory()

    for layer_idx, layer in enumerate(layers):
        for family in SITE_FAMILIES:
            clear_memory()

            if family == "attn_heads":
                with model.trace(clean_inputs):
                    clean_proxy = attribution_proxy_for_family(layer, family)
                    clean_out = clean_proxy.save()

                with model.trace(corrupted_inputs):
                    corrupted_proxy = attribution_proxy_for_family(layer, family)
                    corrupted_out = corrupted_proxy.save()
                    corrupted_grad = corrupted_proxy.grad.save()
                    logits = model.lm_head.output
                    value = teacher_forced_metric(logits, corrupted_batch)
                    traced_value = value.save()
                    value.backward()

                raw_attr = (
                    saved_value(corrupted_grad).float()
                    * (saved_value(clean_out).float() - saved_value(corrupted_out).float())
                )
                objective_value = float(saved_value(traced_value).item())
                grad_norm = float(saved_value(corrupted_grad).float().norm().item())
                site_attrs = einops.reduce(
                    raw_attr,
                    "batch pos (head d_head) -> head",
                    "sum",
                    head=n_heads,
                    d_head=head_dim,
                ).detach().float().cpu()
                for head_idx, attr_value in enumerate(site_attrs):
                    attribution_sums[site_key("attn_head", layer_idx, head_idx)] += (
                        float(attr_value.item()) * float(chunk_pair_count)
                    )
                attr_abs_sum = float(site_attrs.abs().sum().item())
                attr_max_abs = float(site_attrs.abs().max().item())
                site_count = int(n_heads)
                del clean_out, corrupted_out, corrupted_grad, traced_value, value, raw_attr, site_attrs
            else:
                site_type = "mlp" if family == "mlp" else "resid"
                with torch.inference_mode():
                    with model.trace(clean_inputs):
                        clean_proxy = attribution_proxy_for_family(layer, family)
                        clean_out = clean_proxy.save()

                clean_source = saved_value(clean_out).detach().to("cpu")
                del clean_out
                clear_memory()

                with torch.inference_mode():
                    with model.trace(corrupted_inputs):
                        current = attribution_proxy_for_family(layer, family)
                        current[:, :, :] = to_activation_device(clean_source, current)
                        logits = model.lm_head.output
                        value = teacher_forced_metric(logits, corrupted_batch).save()

                objective_value = float(saved_value(value).item())
                site_attr = torch.tensor(objective_value - chunk_unpatched_value, dtype=torch.float32)
                attribution_sums[site_key(site_type, layer_idx)] += (
                    float(site_attr.item()) * float(chunk_pair_count)
                )
                attr_abs_sum = float(site_attr.abs().item())
                attr_max_abs = float(site_attr.abs().item())
                site_count = 1
                grad_norm = float("nan")
                del clean_source, value, site_attr

            diag_records.append(
                {
                    "chunk_index": int(chunk_idx - 1),
                    "layer": int(layer_idx),
                    "site_family": family,
                    "site_count": int(site_count),
                    "chunk_pairs": int(chunk_pair_count),
                    "objective_value": objective_value,
                    "grad_norm": grad_norm,
                    "attr_abs_sum": attr_abs_sum,
                    "attr_max_abs": attr_max_abs,
                }
            )
            print(
                f"  Layer {layer_idx:02d} | {family} | obj={objective_value:.4f} | "
                f"grad_norm={grad_norm:.4f} | attr_abs_sum={attr_abs_sum:.4f}"
            )

            clear_memory()

    del clean_batch, corrupted_batch, clean_inputs, corrupted_inputs
    clear_memory()

attribution_site_df = pd.DataFrame(
    [
        {
            "site_type": site[0],
            "layer": int(site[1]),
            "head": int(site[2]),
            "site": site_label(site),
            "attribution": float(attr_sum) / float(total_pairs),
        }
        for site, attr_sum in attribution_sums.items()
    ]
)
attribution_site_df["abs_attribution"] = attribution_site_df["attribution"].abs()

patching_results = torch.zeros((n_layers, n_heads), dtype=torch.float32).numpy()
for row in attribution_site_df[attribution_site_df["site_type"] == "attn_head"].itertuples(index=False):
    patching_results[int(row.layer), int(row.head)] = float(row.attribution)

diagnostics_df = pd.DataFrame(diag_records)
layer_summary_df = (
    diagnostics_df.groupby(["site_family", "layer"], as_index=False)[["grad_norm", "attr_abs_sum", "attr_max_abs"]]
    .mean()
    .sort_values(["site_family", "layer"])
    .reset_index(drop=True)
)
display(attribution_site_df.sort_values("abs_attribution", ascending=False).head(30))
display(layer_summary_df)
display(diagnostics_df.head(20))

if "attn_heads" in SITE_FAMILIES:
    fig = px.imshow(
        patching_results,
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0.0,
        title="Commitment attribution patching over attention heads",
        labels={"x": "Head", "y": "Layer", "color": "Attribution"},
    )
    fig.show()


## Circuit validation

Now use actual denoising interventions on the activation-site circuit found above.

Normalized faithfulness follows Hanna et al. (2024):
`(m - b_prime) / (b - b_prime)`, where `m` is the patched circuit metric,
`b` is the clean whole-model metric, and `b_prime` is the corrupted whole-model metric.
The random baseline samples same-size activation-site sets and scores them on the same scale.


In [ ]:
CIRCUIT_SELECT = os.environ.get("ATTR_PATCH_CIRCUIT_SELECT", "positive").strip().lower()
FAITHFULNESS_THRESHOLD = float(os.environ.get("ATTR_PATCH_FAITHFULNESS_THRESHOLD", "0.85"))
MAX_CIRCUIT_SIZE_RAW = os.environ.get("ATTR_PATCH_CIRCUIT_MAX_SIZE", "auto").strip().lower()
RANDOM_CIRCUIT_COUNT = int(os.environ.get("ATTR_PATCH_RANDOM_CIRCUIT_COUNT", "8"))
VALIDATION_MAX_CHUNKS = int(os.environ.get("ATTR_PATCH_VALIDATION_MAX_CHUNKS", "0"))
VALIDATION_SEED = int(os.environ.get("ATTR_PATCH_VALIDATION_SEED", "17"))
ACTIVATION_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

site_df = attribution_site_df.copy()
site_df["head"] = site_df["head"].fillna(NO_HEAD).astype(int)
site_df["abs_attribution"] = site_df["attribution"].abs()

if CIRCUIT_SELECT == "positive" and (site_df["attribution"] > 0).any():
    ranked_site_df = (
        site_df[site_df["attribution"] > 0]
        .sort_values("attribution", ascending=False)
        .reset_index(drop=True)
    )
elif CIRCUIT_SELECT == "negative" and (site_df["attribution"] < 0).any():
    ranked_site_df = (
        site_df[site_df["attribution"] < 0]
        .sort_values("attribution", ascending=True)
        .reset_index(drop=True)
    )
elif CIRCUIT_SELECT in {"positive", "abs"}:
    ranked_site_df = (
        site_df.sort_values("abs_attribution", ascending=False)
        .reset_index(drop=True)
    )
else:
    raise ValueError("ATTR_PATCH_CIRCUIT_SELECT must be 'positive', 'negative', or 'abs'.")

if MAX_CIRCUIT_SIZE_RAW in {"", "auto", "all", "none"}:
    max_circuit_size = len(ranked_site_df)
else:
    max_circuit_size = min(int(MAX_CIRCUIT_SIZE_RAW), len(ranked_site_df))
if max_circuit_size <= 0:
    raise ValueError("No ranked circuit sites are available.")

ranked_site_df = ranked_site_df.head(max_circuit_size).reset_index(drop=True)
ranked_sites = [
    site_key(row.site_type, row.layer, row.head)
    for row in ranked_site_df.itertuples(index=False)
]
all_sites = [
    site_key(row.site_type, row.layer, row.head)
    for row in site_df.itertuples(index=False)
]
rng = np.random.default_rng(VALIDATION_SEED)

display(ranked_site_df.head(20))
print(
    f"Ranked {len(ranked_sites)} candidate site(s) using CIRCUIT_SELECT={CIRCUIT_SELECT!r}; "
    f"target normalized faithfulness >= {FAITHFULNESS_THRESHOLD:.2f}."
)
print(
    f"Candidate search space: {len(ranked_sites)} ranked site(s) out of "
    f"{len(all_sites)} total selected activation site(s)."
)
if CIRCUIT_SELECT == "positive":
    print(
        "Note: top_k with CIRCUIT_SELECT='positive' is not the full model, "
        "and not even necessarily all selected activation sites."
    )


def raw_metric_from_logits(logits, batch):
    return paired_metric_from_logits(logits, batch)


def normalized_faithfulness_from_raw(metric, *, clean_baseline, corrupted_baseline):
    denom = float(clean_baseline) - float(corrupted_baseline)
    if abs(denom) < 1e-8:
        raise ValueError("Clean and corrupted baselines are too close for normalization.")
    return (float(metric) - float(corrupted_baseline)) / denom


def normalize_sites(selected_sites):
    normalized = []
    for site in selected_sites:
        if len(site) == 2:
            layer_idx, head_idx = site
            normalized.append(site_key("attn_head", layer_idx, head_idx))
        else:
            site_type, layer_idx, head_idx = site
            normalized.append(site_key(site_type, layer_idx, head_idx))
    return normalized


def group_sites_by_layer(selected_sites):
    grouped = {"attn_head": {}, "mlp": set(), "resid": set()}
    for site_type, layer_idx, head_idx in normalize_sites(selected_sites):
        if site_type == "attn_head":
            grouped["attn_head"].setdefault(layer_idx, set()).add(head_idx)
        elif site_type in {"mlp", "resid"}:
            grouped[site_type].add(layer_idx)
        else:
            raise ValueError(f"Unsupported site type: {site_type}")
    grouped["attn_head"] = {
        layer_idx: sorted(heads)
        for layer_idx, heads in grouped["attn_head"].items()
    }
    grouped["mlp"] = sorted(grouped["mlp"])
    grouped["resid"] = sorted(grouped["resid"])
    return grouped


def validation_chunks(chunks):
    if VALIDATION_MAX_CHUNKS > 0:
        return chunks[:VALIDATION_MAX_CHUNKS]
    return chunks


def score_unpatched_chunks(chunks):
    scores = []
    for chunk_pairs in chunks:
        _, corrupted_batch, _, corrupted_inputs = build_batches_for_pairs(chunk_pairs)
        with torch.inference_mode():
            logits = model.trace(corrupted_inputs, trace=False).logits
        metric = raw_metric_from_logits(logits, corrupted_batch)
        scores.append(float(metric.item()))
        del logits, metric, corrupted_batch, corrupted_inputs
        clear_memory()
    return scores


def score_clean_corrupted_baselines(chunks, *, return_chunk_records=False):
    clean_total = 0.0
    corrupted_total = 0.0
    weight_total = 0
    chunk_records = []
    for chunk_idx, chunk_pairs in enumerate(chunks):
        clean_batch, corrupted_batch, clean_inputs, corrupted_inputs = build_batches_for_pairs(chunk_pairs)
        weight = len(chunk_pairs)
        with torch.inference_mode():
            clean_logits = model.trace(clean_inputs, trace=False).logits
        clean_metric = raw_metric_from_logits(clean_logits, clean_batch)
        clean_value = float(clean_metric.item())
        clean_total += clean_value * weight
        del clean_logits, clean_metric
        clear_memory()

        with torch.inference_mode():
            corrupted_logits = model.trace(corrupted_inputs, trace=False).logits
        corrupted_metric = raw_metric_from_logits(corrupted_logits, corrupted_batch)
        corrupted_value = float(corrupted_metric.item())
        corrupted_total += corrupted_value * weight
        del corrupted_logits, corrupted_metric
        clear_memory()

        weight_total += weight
        chunk_records.append(
            {
                "chunk_index": int(chunk_idx),
                "n_pairs": int(weight),
                "clean_metric": clean_value,
                "corrupted_metric": corrupted_value,
                "denominator": clean_value - corrupted_value,
            }
        )
        del clean_batch, corrupted_batch, clean_inputs, corrupted_inputs
        clear_memory()
    clean_baseline = clean_total / weight_total
    corrupted_baseline = corrupted_total / weight_total
    if return_chunk_records:
        return clean_baseline, corrupted_baseline, chunk_records
    return clean_baseline, corrupted_baseline


def patch_circuit_chunk(chunk_pairs, selected_sites):
    grouped = group_sites_by_layer(selected_sites)
    if not normalize_sites(selected_sites):
        raise ValueError("No circuit sites were selected.")

    clean_batch, corrupted_batch, clean_inputs, corrupted_inputs = build_batches_for_pairs(chunk_pairs)
    clean_proxies = {}
    with torch.inference_mode():
        with model.trace(clean_inputs):
            for layer_idx in grouped["attn_head"]:
                clean_proxies[site_key("attn_head", layer_idx)] = attn_out_input(layers[layer_idx]).save()
            for layer_idx in grouped["mlp"]:
                clean_proxies[site_key("mlp", layer_idx)] = mlp_output(layers[layer_idx]).save()
            for layer_idx in grouped["resid"]:
                clean_proxies[site_key("resid", layer_idx)] = resid_output(layers[layer_idx]).save()

    clean_sources = {
        source_key: saved_value(proxy).detach().to("cpu")
        for source_key, proxy in clean_proxies.items()
    }
    del clean_proxies, clean_batch, clean_inputs
    clear_memory()

    with torch.inference_mode():
        with model.trace(corrupted_inputs):
            for layer_idx, heads in grouped["attn_head"].items():
                current = attn_out_input(layers[layer_idx])
                source = clean_sources[site_key("attn_head", layer_idx)]
                for head_idx in heads:
                    start = int(head_idx) * head_dim
                    stop = start + head_dim
                    current[:, :, start:stop] = to_activation_device(
                        source[:, :, start:stop],
                        current[:, :, start:stop],
                    )
            for layer_idx in grouped["mlp"]:
                current = mlp_output(layers[layer_idx])
                source = clean_sources[site_key("mlp", layer_idx)]
                current[:, :, :] = to_activation_device(source, current)
            for layer_idx in grouped["resid"]:
                current = resid_output(layers[layer_idx])
                source = clean_sources[site_key("resid", layer_idx)]
                current[:, :, :] = to_activation_device(source, current)
            logits = model.lm_head.output
            metric = raw_metric_from_logits(logits, corrupted_batch).save()

    value = float(saved_value(metric).item())
    del metric, corrupted_batch, corrupted_inputs, clean_sources
    clear_memory()
    return value


def score_circuit_on_chunks(
    selected_sites,
    chunks,
    *,
    clean_baseline,
    corrupted_baseline,
    unpatched_scores=None,
    baseline_chunk_records=None,
    label="circuit",
):
    chunks = validation_chunks(chunks)
    if unpatched_scores is None:
        unpatched_scores = score_unpatched_chunks(chunks)
    patched_total = 0.0
    unpatched_total = 0.0
    weight_total = 0
    for chunk_idx, chunk_pairs in enumerate(chunks):
        patched = patch_circuit_chunk(chunk_pairs, selected_sites)
        unpatched = float(unpatched_scores[chunk_idx])
        weight = len(chunk_pairs)
        patched_total += patched * weight
        unpatched_total += unpatched * weight
        weight_total += weight
        chunk_faithfulness = None
        if baseline_chunk_records is not None:
            chunk_record = baseline_chunk_records[chunk_idx]
            if abs(float(chunk_record["denominator"])) >= 1e-8:
                chunk_faithfulness = normalized_faithfulness_from_raw(
                    patched,
                    clean_baseline=float(chunk_record["clean_metric"]),
                    corrupted_baseline=float(chunk_record["corrupted_metric"]),
                )
        chunk_faithfulness_text = (
            ""
            if chunk_faithfulness is None
            else f" | chunk_faithfulness={chunk_faithfulness:.4f}"
        )
        print(
            f"{label} | chunk {chunk_idx + 1}/{len(chunks)} | "
            f"b_prime={unpatched:.4f} | m={patched:.4f}{chunk_faithfulness_text}"
        )
    patched_metric = patched_total / weight_total
    unpatched_metric = unpatched_total / weight_total
    normalized_faithfulness = normalized_faithfulness_from_raw(
        patched_metric,
        clean_baseline=clean_baseline,
        corrupted_baseline=corrupted_baseline,
    )
    unpatched_faithfulness = normalized_faithfulness_from_raw(
        unpatched_metric,
        clean_baseline=clean_baseline,
        corrupted_baseline=corrupted_baseline,
    )
    return {
        "unpatched_metric": unpatched_metric,
        "patched_metric": patched_metric,
        "clean_baseline_metric": float(clean_baseline),
        "corrupted_baseline_metric": float(corrupted_baseline),
        "unpatched_faithfulness": unpatched_faithfulness,
        "normalized_faithfulness": normalized_faithfulness,
        "faithfulness_lift": normalized_faithfulness - unpatched_faithfulness,
        "n_chunks": len(chunks),
        "n_pairs": int(sum(len(chunk) for chunk in chunks)),
        "circuit_size": len(normalize_sites(selected_sites)),
    }


def top_ranked_sites(edge_count):
    edge_count = int(edge_count)
    if edge_count <= 0:
        raise ValueError("edge_count must be positive.")
    if edge_count > len(ranked_sites):
        raise ValueError(f"edge_count={edge_count} exceeds {len(ranked_sites)} ranked sites.")
    return ranked_sites[:edge_count]


def binary_search_min_faithful_circuit(
    chunks,
    *,
    clean_baseline,
    corrupted_baseline,
    unpatched_scores,
    threshold,
    max_size,
    baseline_chunk_records=None,
):
    score_cache = {}
    search_records = []

    def evaluate(edge_count):
        edge_count = int(edge_count)
        if edge_count not in score_cache:
            record = score_circuit_on_chunks(
                top_ranked_sites(edge_count),
                chunks,
                clean_baseline=clean_baseline,
                corrupted_baseline=corrupted_baseline,
                unpatched_scores=unpatched_scores,
                baseline_chunk_records=baseline_chunk_records,
                label=f"top_{edge_count}",
            )
            record = {
                "candidate_edge_count": edge_count,
                "meets_threshold": bool(record["normalized_faithfulness"] >= float(threshold)),
                **record,
            }
            score_cache[edge_count] = record
            search_records.append({"search_step": len(search_records), **record})
        return score_cache[edge_count]

    low = 1
    high = int(max_size)
    best_record = None
    while low <= high:
        mid = (low + high) // 2
        mid_record = evaluate(mid)
        if mid_record["meets_threshold"]:
            best_record = mid_record
            high = mid - 1
        else:
            low = mid + 1

    if best_record is None:
        best_record = evaluate(max_size)
        status = "threshold_not_reached"
    else:
        status = "threshold_reached"

    search_df = pd.DataFrame(search_records).sort_values("search_step").reset_index(drop=True)
    return int(best_record["candidate_edge_count"]), best_record, search_df, status


def whole_model_reference_record(circuit_id, raw_metric):
    normalized = normalized_faithfulness_from_raw(
        raw_metric,
        clean_baseline=VALIDATION_CLEAN_BASELINE,
        corrupted_baseline=VALIDATION_CORRUPTED_BASELINE,
    )
    return {
        "circuit_kind": "whole_model",
        "circuit_id": circuit_id,
        "search_status": "reference",
        "faithfulness_threshold": FAITHFULNESS_THRESHOLD,
        "candidate_edge_count": 0,
        "meets_threshold": bool(normalized >= FAITHFULNESS_THRESHOLD),
        "unpatched_metric": VALIDATION_CORRUPTED_BASELINE,
        "patched_metric": float(raw_metric),
        "clean_baseline_metric": VALIDATION_CLEAN_BASELINE,
        "corrupted_baseline_metric": VALIDATION_CORRUPTED_BASELINE,
        "unpatched_faithfulness": 0.0,
        "normalized_faithfulness": normalized,
        "faithfulness_lift": normalized,
        "n_chunks": len(baseline_chunks),
        "n_pairs": int(sum(len(chunk) for chunk in baseline_chunks)),
        "circuit_size": 0,
        "ranked_candidate_count": len(ranked_sites),
        "total_candidate_sites": len(all_sites),
    }


In [ ]:
baseline_chunks = validation_chunks(pair_chunks)
VALIDATION_CLEAN_BASELINE, VALIDATION_CORRUPTED_BASELINE, validation_baseline_chunk_records = score_clean_corrupted_baselines(
    baseline_chunks,
    return_chunk_records=True,
)
baseline_unpatched_scores = [
    float(record["corrupted_metric"])
    for record in validation_baseline_chunk_records
]
validation_baseline_df = pd.DataFrame(validation_baseline_chunk_records)
display(validation_baseline_df)
print(f"Validation clean baseline b: {VALIDATION_CLEAN_BASELINE:.4f}")
print(f"Validation corrupted baseline b_prime: {VALIDATION_CORRUPTED_BASELINE:.4f}")

discovered_edge_count, discovered_record, circuit_search_df, circuit_search_status = (
    binary_search_min_faithful_circuit(
        baseline_chunks,
        clean_baseline=VALIDATION_CLEAN_BASELINE,
        corrupted_baseline=VALIDATION_CORRUPTED_BASELINE,
        unpatched_scores=baseline_unpatched_scores,
        baseline_chunk_records=validation_baseline_chunk_records,
        threshold=FAITHFULNESS_THRESHOLD,
        max_size=len(ranked_sites),
    )
)
discovered_sites = top_ranked_sites(discovered_edge_count)
discovered_circuit_df = ranked_site_df.head(discovered_edge_count).reset_index(drop=True)

display(circuit_search_df)
display(discovered_circuit_df)
print(
    f"Selected {discovered_edge_count} activation site(s): {circuit_search_status}; "
    f"normalized_faithfulness={discovered_record['normalized_faithfulness']:.4f}; "
    f"threshold={FAITHFULNESS_THRESHOLD:.2f}."
)

baseline_records = [
    whole_model_reference_record("corrupted_b_prime", VALIDATION_CORRUPTED_BASELINE),
    whole_model_reference_record("clean_b", VALIDATION_CLEAN_BASELINE),
]
baseline_records.append(
    {
        "circuit_kind": "discovered",
        "circuit_id": f"top_{discovered_edge_count}",
        "search_status": circuit_search_status,
        "faithfulness_threshold": FAITHFULNESS_THRESHOLD,
        "ranked_candidate_count": len(ranked_sites),
        "total_candidate_sites": len(all_sites),
        **discovered_record,
    }
)

for random_idx in range(RANDOM_CIRCUIT_COUNT):
    random_site_indices = rng.choice(len(all_sites), size=len(discovered_sites), replace=False)
    random_sites = [all_sites[int(idx)] for idx in random_site_indices]
    random_record = score_circuit_on_chunks(
        random_sites,
        baseline_chunks,
        clean_baseline=VALIDATION_CLEAN_BASELINE,
        corrupted_baseline=VALIDATION_CORRUPTED_BASELINE,
        unpatched_scores=baseline_unpatched_scores,
        baseline_chunk_records=validation_baseline_chunk_records,
        label=f"random_{random_idx:02d}",
    )
    baseline_records.append(
        {
            "circuit_kind": "random",
            "circuit_id": f"random_{random_idx:02d}",
            "search_status": "same_size_random",
            "faithfulness_threshold": FAITHFULNESS_THRESHOLD,
            "candidate_edge_count": len(discovered_sites),
            "meets_threshold": bool(random_record["normalized_faithfulness"] >= FAITHFULNESS_THRESHOLD),
            "ranked_candidate_count": len(ranked_sites),
            "total_candidate_sites": len(all_sites),
            **random_record,
        }
    )

random_baseline_df = pd.DataFrame(baseline_records)
display(random_baseline_df)
comparison_df = random_baseline_df[
    random_baseline_df["circuit_kind"].isin(["discovered", "random"])
].copy()
display(
    comparison_df.groupby("circuit_kind", as_index=False)["normalized_faithfulness"]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)
fig = px.strip(
    comparison_df,
    x="circuit_kind",
    y="normalized_faithfulness",
    color="circuit_kind",
    title="Normalized faithfulness: discovered circuit vs same-size random circuits",
)
fig.show()


In [ ]:
CROSS_CORPUS_ENVS = [
    env.strip()
    for env in os.environ.get(
        "ATTR_PATCH_CROSS_CORPUS_ENVS",
        "advisor_audit,car_sales,gridworld,interview",
    ).split(",")
    if env.strip()
]
CROSS_CORPUS_PAIR_COUNT = int(os.environ.get("ATTR_PATCH_CROSS_CORPUS_PAIR_COUNT", "3"))
CROSS_CORPUS_SEARCH_LIMIT = int(os.environ.get("ATTR_PATCH_CROSS_CORPUS_SEARCH_LIMIT", "128"))


def count_alpha_words(text):
    return len(re.findall(r"[A-Za-z]+", str(text)))


def usable_sentence(text):
    text = str(text).strip()
    if not text:
        return False
    if EXCLUDE_MULTILINE_SENTENCES and "\n" in text:
        return False
    if MIN_SENTENCE_ALPHA_WORDS > 0 and count_alpha_words(text) < MIN_SENTENCE_ALPHA_WORDS:
        return False
    return True


def choose_generic_truthful_donor(shared_entry, target_sentence):
    target_norm = ap.normalize_sentence_for_compare(target_sentence)
    candidates = []
    for gen_idx, generation in enumerate(shared_entry.get("generations") or []):
        if generation.get("is_truthful") is not True:
            continue
        first_sentence, _ = ap.extract_first_sentence(generation.get("gen_text", ""))
        first_sentence = str(first_sentence).strip()
        if not usable_sentence(first_sentence):
            continue
        if ap.normalize_sentence_for_compare(first_sentence) == target_norm:
            continue
        candidates.append(
            {
                "gen_idx": int(gen_idx),
                "first_sentence": first_sentence,
                "prompt": generation.get("prompt", shared_entry.get("prompt", "")),
                "prefix_text": generation.get("prefix_text", shared_entry.get("prefix_text", "")),
                "full_generation_text": generation.get("full_generation_text", ""),
                "is_truthful": generation.get("is_truthful"),
                "deceptive": generation.get("deceptive"),
                "parse_error": generation.get("parse_error"),
                "evaluation": generation.get("evaluation"),
                "word_count": count_alpha_words(first_sentence),
            }
        )
    if not candidates:
        return None, []
    candidates = sorted(candidates, key=lambda row: (abs(row["word_count"] - 12), len(row["first_sentence"])))
    return candidates[0], candidates


def load_generic_commitment_pairs_for_env(env_name, *, pair_count, search_limit):
    localization_dir = REPO_ROOT / "DatasetMain" / env_name / MODEL_ID / "localization"
    if not localization_dir.exists():
        print(f"Skipping {env_name}: missing {localization_dir}")
        return pd.DataFrame()

    rows = []
    paths = sorted(localization_dir.glob("sentence_localization_*.json"))
    for path in paths:
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        history = payload.get("history") or []
        if len(history) < 2:
            continue
        example_id = str(payload.get("example_id", path.stem))
        for right_pos in range(1, len(history)):
            left_pos = right_pos - 1
            shared_entry = history[left_pos]
            target_entry = history[right_pos]
            try:
                shared_rate = float(shared_entry.get("deception_rate", float("nan")))
                target_rate = float(target_entry.get("deception_rate", float("nan")))
                commitment_delta = target_rate - shared_rate
            except Exception:
                continue
            if not math.isfinite(commitment_delta) or commitment_delta <= MIN_COMMITMENT_DELTA:
                continue
            if target_rate < MIN_COMMITMENT_DECEPTION_RATE:
                continue

            shared_num_valid = int(shared_entry.get("num_valid") or 0)
            target_num_valid = int(target_entry.get("num_valid") or 0)
            if MIN_NUM_VALID > 0 and (
                shared_num_valid < MIN_NUM_VALID or target_num_valid < MIN_NUM_VALID
            ):
                continue

            deceptive_sentence = str(target_entry.get("sentence_text", "")).strip()
            if not usable_sentence(deceptive_sentence):
                continue
            donor_row, donor_candidates = choose_generic_truthful_donor(shared_entry, deceptive_sentence)
            if donor_row is None:
                continue

            prompt = str(target_entry.get("prompt", payload.get("prompt", "")))
            shared_context_text = str(shared_entry.get("prefix_text", ""))
            truthful_sentence = str(donor_row["first_sentence"]).strip()
            if not prompt or not shared_context_text:
                continue

            rows.append(
                {
                    "pair_id": (
                        f"{env_name}__{ap.slugify(example_id)}__sent_{int(right_pos)}__"
                        f"donor_{int(donor_row['gen_idx'])}"
                    ),
                    "localization_path": str(path),
                    "example_id": example_id,
                    "shared_context_sentence_pos": int(left_pos),
                    "commitment_sentence_pos": int(right_pos),
                    "shared_context_sentence_text": str(shared_entry.get("sentence_text", "")),
                    "deceptive_commitment_sentence": deceptive_sentence,
                    "truthful_donor_sentence": truthful_sentence,
                    "prompt": prompt,
                    "shared_context_text": shared_context_text,
                    "shared_context_deception_rate": shared_rate,
                    "deceptive_prefix_deception_rate": target_rate,
                    "commitment_deception_rate": target_rate,
                    "commitment_delta": commitment_delta,
                    "shared_context_num_valid": shared_num_valid,
                    "shared_context_num_truthful": shared_entry.get("num_truthful"),
                    "deceptive_prefix_num_valid": target_num_valid,
                    "deceptive_prefix_num_truthful": target_entry.get("num_truthful"),
                    "donor_generation_idx": int(donor_row["gen_idx"]),
                    "donor_full_generation_text": donor_row.get("full_generation_text", ""),
                    "donor_is_truthful": donor_row.get("is_truthful") is True,
                    "donor_deceptive": donor_row.get("deceptive"),
                    "donor_parse_error": donor_row.get("parse_error"),
                    "donor_evaluation": ap.to_json_safe(donor_row.get("evaluation")),
                    "donor_clarity_score": float(donor_row["word_count"]),
                    "n_truthful_donors": int(len(donor_candidates)),
                    "shared_prefix_text": prompt + shared_context_text,
                    "deceptive_branch_text": prompt + ap.append_continuation(
                        shared_context_text,
                        deceptive_sentence,
                    ),
                    "truthful_branch_text": prompt + ap.append_continuation(
                        shared_context_text,
                        truthful_sentence,
                    ),
                }
            )
            if len(rows) > max(search_limit * 4, search_limit + 100):
                rows = sorted(
                    rows,
                    key=lambda row: (
                        row["commitment_delta"],
                        row["deceptive_prefix_deception_rate"],
                        row["n_truthful_donors"],
                        row["donor_clarity_score"],
                    ),
                    reverse=True,
                )[:search_limit]

    if not rows:
        return pd.DataFrame()
    rows = sorted(
        rows,
        key=lambda row: (
            row["commitment_delta"],
            row["deceptive_prefix_deception_rate"],
            row["n_truthful_donors"],
            row["donor_clarity_score"],
        ),
        reverse=True,
    )[:pair_count]
    df = pd.DataFrame(rows).reset_index(drop=True)
    df.insert(0, "pair_index", range(len(df)))
    return df


def prepare_records_without_length_cap(pairs):
    global MAX_INPUT_TOKENS
    previous_max_input_tokens = MAX_INPUT_TOKENS
    try:
        MAX_INPUT_TOKENS = None
        return prepare_pair_records(pairs)
    finally:
        MAX_INPUT_TOKENS = previous_max_input_tokens


cross_records = []
for env_name in CROSS_CORPUS_ENVS:
    env_pairs_df = load_generic_commitment_pairs_for_env(
        env_name,
        pair_count=CROSS_CORPUS_PAIR_COUNT,
        search_limit=CROSS_CORPUS_SEARCH_LIMIT,
    )
    if env_pairs_df.empty:
        cross_records.append(
            {
                "environment": env_name,
                "status": "no_pairs",
                "n_pairs": 0,
            }
        )
        continue

    env_prepared_pairs = prepare_records_without_length_cap(env_pairs_df)
    env_pair_chunks = validation_chunks(pair_chunk_slices(env_prepared_pairs, BATCH_PAIR_COUNT))
    env_clean_baseline, env_corrupted_baseline, env_baseline_chunk_records = score_clean_corrupted_baselines(
        env_pair_chunks,
        return_chunk_records=True,
    )
    env_unpatched_scores = [
        float(record["corrupted_metric"])
        for record in env_baseline_chunk_records
    ]
    env_record = score_circuit_on_chunks(
        discovered_sites,
        env_pair_chunks,
        clean_baseline=env_clean_baseline,
        corrupted_baseline=env_corrupted_baseline,
        unpatched_scores=env_unpatched_scores,
        baseline_chunk_records=env_baseline_chunk_records,
        label=f"{env_name}_discovered",
    )
    cross_records.append(
        {
            "environment": env_name,
            "status": "ok",
            "clean_baseline": env_clean_baseline,
            "corrupted_baseline": env_corrupted_baseline,
            "max_total_len": max(int(pair["max_total_len"]) for pair in env_prepared_pairs),
            **env_record,
        }
    )
    clear_memory()

cross_corpus_df = pd.DataFrame(cross_records)
display(cross_corpus_df)


## Circuit steering

Attribution patching tells us *where* to intervene. The matched truthful-minus-deceptive
activation differences tell us *which direction* to intervene in.

For each selected activation site, estimate `d = E[z_truthful - z_deceptive]` on the
matched pairs. During generation, add `alpha * d` at the selected site, so the steering
intervention is reusable and does not need a donor example.


In [ ]:
STEERING_ALPHA = float(os.environ.get("ATTR_PATCH_STEERING_ALPHA", "1.0"))
STEERING_GENERATION_COUNT = int(os.environ.get("ATTR_PATCH_STEERING_GENERATION_COUNT", "10"))
STEERING_MAX_NEW_TOKENS = int(os.environ.get("ATTR_PATCH_STEERING_MAX_NEW_TOKENS", "96"))
STEERING_TEMPERATURE = float(os.environ.get("ATTR_PATCH_STEERING_TEMPERATURE", "0.7"))
STEERING_TOP_P = float(os.environ.get("ATTR_PATCH_STEERING_TOP_P", "0.95"))
STEERING_SEED = int(os.environ.get("ATTR_PATCH_STEERING_SEED", "23"))
STEERING_INCLUDE_BASELINE = os.environ.get("ATTR_PATCH_STEERING_INCLUDE_BASELINE", "0") == "1"
STEERING_POSITION = os.environ.get("ATTR_PATCH_STEERING_POSITION", "last").strip().lower()

if STEERING_POSITION not in {"last", "all"}:
    raise ValueError("ATTR_PATCH_STEERING_POSITION must be 'last' or 'all'.")


def steering_proxy_for_site(site_type, layer_idx):
    if site_type == "attn_head":
        return attn_out_input(layers[layer_idx])
    if site_type == "mlp":
        return mlp_output(layers[layer_idx])
    if site_type == "resid":
        return resid_output(layers[layer_idx])
    raise ValueError(f"Unsupported steering site type: {site_type}")


def compute_site_steering_vectors(chunks, selected_sites):
    grouped = group_sites_by_layer(selected_sites)
    sites = normalize_sites(selected_sites)
    sums = {}
    for site_type, layer_idx, head_idx in sites:
        if site_type == "attn_head":
            sums[site_key(site_type, layer_idx, head_idx)] = torch.zeros(head_dim, dtype=torch.float32)
        else:
            sums[site_key(site_type, layer_idx)] = torch.zeros(model.config.hidden_size, dtype=torch.float32)
    counts = {site: 0 for site in sums}

    capture_specs = []
    for layer_idx in grouped["attn_head"]:
        capture_specs.append(site_key("attn_head", layer_idx))
    for layer_idx in grouped["mlp"]:
        capture_specs.append(site_key("mlp", layer_idx))
    for layer_idx in grouped["resid"]:
        capture_specs.append(site_key("resid", layer_idx))

    for capture_site in capture_specs:
        site_type, layer_idx, _head_idx = capture_site
        selected_heads = grouped["attn_head"].get(layer_idx, []) if site_type == "attn_head" else [NO_HEAD]
        print(f"Computing steering directions for {site_label(capture_site)} ({len(selected_heads)} site(s))")
        for chunk_pairs in chunks:
            clean_batch, _, clean_inputs, _ = build_batches_for_pairs(chunk_pairs)
            with torch.inference_mode():
                with model.trace(clean_inputs):
                    layer_proxy = steering_proxy_for_site(site_type, layer_idx).save()
            layer_acts = saved_value(layer_proxy).detach().float().cpu()

            for local_pair_idx, _pair in enumerate(chunk_pairs):
                row_start = 2 * local_pair_idx
                row_indices_by_role = {
                    str(clean_batch["rows"][row_start + offset]["branch_role"]): row_start + offset
                    for offset in range(2)
                }
                deceptive_row_idx = int(row_indices_by_role["deceptive"])
                truthful_row_idx = int(row_indices_by_role["truthful"])
                deceptive_row = clean_batch["rows"][deceptive_row_idx]
                truthful_row = clean_batch["rows"][truthful_row_idx]
                deceptive_slice = slice(
                    int(deceptive_row["score_start_pos"]),
                    int(deceptive_row["score_stop_pos"]),
                )
                truthful_slice = slice(
                    int(truthful_row["score_start_pos"]),
                    int(truthful_row["score_stop_pos"]),
                )
                for head_idx in selected_heads:
                    if site_type == "attn_head":
                        start = int(head_idx) * head_dim
                        stop = start + head_dim
                        deceptive_vec = layer_acts[
                            deceptive_row_idx,
                            deceptive_slice,
                            start:stop,
                        ].mean(dim=0)
                        truthful_vec = layer_acts[
                            truthful_row_idx,
                            truthful_slice,
                            start:stop,
                        ].mean(dim=0)
                        site = site_key("attn_head", layer_idx, head_idx)
                    else:
                        deceptive_vec = layer_acts[
                            deceptive_row_idx,
                            deceptive_slice,
                            :,
                        ].mean(dim=0)
                        truthful_vec = layer_acts[
                            truthful_row_idx,
                            truthful_slice,
                            :,
                        ].mean(dim=0)
                        site = site_key(site_type, layer_idx)
                    sums[site] += truthful_vec - deceptive_vec
                    counts[site] += 1

            del layer_proxy, layer_acts, clean_batch, clean_inputs
            clear_memory()

    vectors = {
        site: sums[site] / max(int(counts[site]), 1)
        for site in sums
    }
    vector_df = pd.DataFrame(
        [
            {
                "site_type": site[0],
                "layer": int(site[1]),
                "head": int(site[2]),
                "site": site_label(site),
                "n_pairs": int(counts[site]),
                "direction_norm": float(vector.norm().item()),
                "direction_mean_abs": float(vector.abs().mean().item()),
            }
            for site, vector in vectors.items()
        ]
    ).sort_values(["site_type", "layer", "head"]).reset_index(drop=True)
    return vectors, vector_df


steering_vectors, steering_vector_df = compute_site_steering_vectors(pair_chunks, discovered_sites)
display(steering_vector_df)
print(
    f"Computed {len(steering_vectors)} activation-site steering vector(s) "
    f"for alpha={STEERING_ALPHA:.3f}."
)


In [ ]:
def sample_next_token_from_logits(logits, *, temperature, top_p):
    logits = logits.detach().float().cpu()
    if float(temperature) <= 0:
        return torch.argmax(logits, dim=-1, keepdim=True)

    logits = logits / float(temperature)
    if 0 < float(top_p) < 1:
        sorted_logits, sorted_indices = torch.sort(logits, descending=True, dim=-1)
        sorted_probs = torch.softmax(sorted_logits, dim=-1)
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
        remove_mask = cumulative_probs > float(top_p)
        remove_mask[..., 1:] = remove_mask[..., :-1].clone()
        remove_mask[..., 0] = False
        sorted_logits = sorted_logits.masked_fill(remove_mask, float("-inf"))
        filtered_logits = torch.full_like(logits, float("-inf"))
        logits = filtered_logits.scatter(dim=-1, index=sorted_indices, src=sorted_logits)

    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)


def steering_position_slice(current):
    if STEERING_POSITION == "last":
        return (slice(None), slice(-1, None))
    return (slice(None), slice(None))


def add_steering_direction(activation_slice, direction):
    direction = direction.to(device=activation_slice.device, dtype=activation_slice.dtype)
    return activation_slice + direction.view(1, 1, -1)


def generate_with_site_steering(prompt_text, *, seed, do_steer):
    torch.manual_seed(int(seed))
    encoded = encode_text_for_model(tokenizer, prompt_text, max_input_tokens=None)
    current_ids = encoded["input_ids"].clone()
    prompt_token_count = int(current_ids.shape[1])
    generated_ids = []
    ended_with_eos = False

    grouped_vectors = group_sites_by_layer(discovered_sites)
    scaled_vectors = {
        site: (float(STEERING_ALPHA) * vector).detach().cpu()
        for site, vector in steering_vectors.items()
    }

    for _step in range(STEERING_MAX_NEW_TOKENS):
        inputs = {
            "input_ids": current_ids,
            "attention_mask": torch.ones_like(current_ids),
        }
        if do_steer:
            with torch.inference_mode():
                with model.trace(inputs):
                    for layer_idx, heads in grouped_vectors["attn_head"].items():
                        current = attn_out_input(layers[layer_idx])
                        batch_slice, pos_slice = steering_position_slice(current)
                        for head_idx in heads:
                            site = site_key("attn_head", layer_idx, head_idx)
                            start = int(head_idx) * head_dim
                            stop = start + head_dim
                            current[batch_slice, pos_slice, start:stop] = (
                                add_steering_direction(
                                    current[batch_slice, pos_slice, start:stop],
                                    scaled_vectors[site],
                                )
                            )
                    for layer_idx in grouped_vectors["mlp"]:
                        current = mlp_output(layers[layer_idx])
                        batch_slice, pos_slice = steering_position_slice(current)
                        site = site_key("mlp", layer_idx)
                        current[batch_slice, pos_slice, :] = add_steering_direction(
                            current[batch_slice, pos_slice, :],
                            scaled_vectors[site],
                        )
                    for layer_idx in grouped_vectors["resid"]:
                        current = resid_output(layers[layer_idx])
                        batch_slice, pos_slice = steering_position_slice(current)
                        site = site_key("resid", layer_idx)
                        current[batch_slice, pos_slice, :] = add_steering_direction(
                            current[batch_slice, pos_slice, :],
                            scaled_vectors[site],
                        )
                    next_logits = model.lm_head.output[:, -1, :].save()
            logits = saved_value(next_logits)
        else:
            with torch.inference_mode():
                logits = model.trace(inputs, trace=False).logits[:, -1, :]

        next_token = sample_next_token_from_logits(
            logits,
            temperature=STEERING_TEMPERATURE,
            top_p=STEERING_TOP_P,
        ).to(dtype=current_ids.dtype)
        token_id = int(next_token.item())
        generated_ids.append(token_id)
        current_ids = torch.cat([current_ids, next_token.cpu()], dim=1)
        del logits, next_token, inputs
        clear_memory()

        if tokenizer.eos_token_id is not None and token_id == int(tokenizer.eos_token_id):
            ended_with_eos = True
            break

    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
    full_text = tokenizer.decode(current_ids[0], skip_special_tokens=True)
    return {
        "generated_text": generated_text,
        "full_text": full_text,
        "prompt_token_count": prompt_token_count,
        "n_new_tokens": len(generated_ids),
        "ended_with_eos": ended_with_eos,
        "hit_token_cap": len(generated_ids) >= int(STEERING_MAX_NEW_TOKENS) and not ended_with_eos,
    }


steering_prompt_rows = [
    {
        "pair_index": int(pair["pair_index"]),
        "example_id": str(pair["example_id"]),
        "prompt_text": str(pair["shared_prefix_text"]),
        "deceptive_commitment_sentence": str(pair["deceptive_commitment_sentence"]),
        "truthful_donor_sentence": str(pair["truthful_donor_sentence"]),
    }
    for pair in sorted(prepared_pairs, key=lambda row: int(row["pair_index"]))[:STEERING_GENERATION_COUNT]
]

steering_generation_records = []
for sample_idx, prompt_row in enumerate(steering_prompt_rows):
    seed = STEERING_SEED + sample_idx
    if STEERING_INCLUDE_BASELINE:
        baseline = generate_with_site_steering(
            prompt_row["prompt_text"],
            seed=seed,
            do_steer=False,
        )
        steering_generation_records.append(
            {
                **prompt_row,
                "sample_index": int(sample_idx),
                "condition": "baseline",
                "alpha": 0.0,
                "seed": int(seed),
                **baseline,
            }
        )

    steered = generate_with_site_steering(
        prompt_row["prompt_text"],
        seed=seed,
        do_steer=True,
    )
    steering_generation_records.append(
        {
            **prompt_row,
            "sample_index": int(sample_idx),
            "condition": "steered",
            "alpha": float(STEERING_ALPHA),
            "seed": int(seed),
            **steered,
        }
    )
    print(
        f"Generated {sample_idx + 1}/{len(steering_prompt_rows)} | "
        f"condition=steered | n_new_tokens={steered['n_new_tokens']}"
    )

steering_generations_df = pd.DataFrame(steering_generation_records)
display(
    steering_generations_df[
        [
            "sample_index",
            "condition",
            "alpha",
            "n_new_tokens",
            "hit_token_cap",
            "deceptive_commitment_sentence",
            "truthful_donor_sentence",
            "generated_text",
        ]
    ]
)
